# День 5 · RAGAS: готовые метрики оценки RAG

`evaluate()` из RAGAS сам гоняет свою модель-судью по каждому вопросу прогона и считает несколько
метрик за один проход. `judge` (`LangchainLLMWrapper`) и `emb` (`LangchainEmbeddingsWrapper`) — это
просто обёртки над теми же клиентами, что и всюду в курсе, RAGAS требует именно такой интерфейс.

In [ ]:
"""Считает три готовые метрики RAGAS по сохранённому прогону experiment.py и пишет их в Langfuse."""
import json
from pathlib import Path

import labkit
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langfuse import get_client
from ragas import EvaluationDataset, evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import Faithfulness, LLMContextPrecisionWithoutReference, LLMContextRecall, ResponseRelevancy

# --- НАСТРОЙКИ: сначала .local/run-dense-v1.json, потом .local/run-hybrid-v1.json ---
RUN_FILE = ".local/run-hybrid-v1.json"     # какой прогон experiment.py оценивать

BASE_URL = labkit.env("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
JUDGE = labkit.env("JUDGE_MODEL", required=True)
KEY = labkit.env("OPENROUTER_API_KEY", required=True)
langfuse = get_client()

path = labkit.ROOT / "day5-evals" / RUN_FILE  # не Path(__file__): этот файл — ноутбук, __file__ там не определён
rows = json.loads(path.read_text(encoding="utf-8"))
if not rows:
    raise ValueError("Пустой прогон")
judge = LangchainLLMWrapper(ChatOpenAI(model=JUDGE, base_url=BASE_URL, api_key=KEY, temperature=0, timeout=120))
emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="openai/text-embedding-3-small", base_url=BASE_URL, api_key=KEY, check_embedding_ctx_length=False))

## Faithfulness, relevancy, context precision — и context recall отдельно

`context_recall` считается только там, где есть `reference` (эталонный ответ человека) — честных
отказов и вопросов без эталона это не касается, поэтому набор для него меньше основного и
результаты аккуратно приджойнены обратно по `user_input`, а не просто пересчитаны средним по всем.

In [ ]:
rows = [r for r in rows if not r.get("unanswerable")]
if not rows:
    raise ValueError("Нет answerable-примеров для RAGAS; отказы оцениваются отдельно")
samples = [{"user_input": r["question"], "response": r["answer"], "retrieved_contexts": r["contexts"], **({"reference": r["reference"]} if r.get("reference") else {})} for r in rows if not r.get("unanswerable")]
with_ref = [s for s in samples if "reference" in s]

result = evaluate(EvaluationDataset.from_list(samples), metrics=[Faithfulness(), ResponseRelevancy(), LLMContextPrecisionWithoutReference()], llm=judge, embeddings=emb)
df = result.to_pandas()
if with_ref:
    df_ref = evaluate(EvaluationDataset.from_list(with_ref), metrics=[LLMContextRecall()], llm=judge).to_pandas()
    df = df.merge(df_ref[["user_input", "context_recall"]], on="user_input", how="left")

skip = {"user_input", "response", "retrieved_contexts", "reference"}
metric_cols = [c for c in df.columns if c not in skip]
print(df[metric_cols].describe().loc[["mean", "min"]].round(3).to_markdown())

## Оценки уходят обратно в те же трейсы

`df` уже в памяти ноутбука после предыдущей ячейки — можно сразу посмотреть `df.head()` отдельной
ячейкой. Ниже — запись метрик обратно в Langfuse, к тем же `trace_id`, что создал `experiment.py`:
так в UI прогон и его метрики качества видны рядом, а не в отдельном отчёте.

In [ ]:
for r, (_, s) in zip(rows, df.iterrows()):
    for name in metric_cols:
        value = s[name]
        if value == value:  # не NaN
            langfuse.create_score(trace_id=r["trace_id"], name=name, value=float(value))
langfuse.flush()
print(f"оценки записаны в {len(rows)} трейсов")